# Day 2 · 04. LangGraph로 RAG 만들기: Naive RAG → 관련성 체크

## 학습 목표
- 03에서 배운 **State · Node · Edge** 문법으로 `retrieve → llm_answer` 두 노드짜리 **Naive RAG** 그래프를 직접 조립한다.
- State에 `relevance` 필드를 추가하고, 검색된 문서가 질문과 관련 있는지 판정하는 **Groundedness Check** 노드를 만들 수 있다.
- **조건부 엣지**로 "관련 있으면 답변, 없으면 재검색" 흐름을 직접 연결할 수 있다.
- 이 구조의 한계(무한 루프)를 실험으로 확인하고, 05 `Self-RAG`가 왜 쿼리 재작성 노드를 끼워 넣는지 이해한다.

## 핵심 키워드
StateGraph · TypedDict · retrieve · llm_answer · Groundedness Check · add_conditional_edges · Recursion Limit

## 이 노트북의 위치
```
03 (LangGraph 문법)   →   04 (그 문법으로 RAG 조립, 관련성 체크)   →   05 (재검색 루프 해결)
```
03은 "LangGraph 자체"를 가르쳐서 여기서는 바로 RAG에 적용한다. 이 노트북 끝에서 부딪히는 *"같은 질문을 다시 넘겨봐야 결과가 같다"* 라는 문제가 바로 05 Self-RAG가 풀어내는 지점이다.

In [ ]:
# 공통 세팅: common 헬퍼를 사용하기 위해 repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '..')

from common import get_chat_model, get_embeddings, provider_badge
from common.loaders import load_any

print(provider_badge())

## 1. 왜 지금 RAG에 LangGraph를 끼워 넣나?

`prompt | llm | parser` 한 줄로도 Naive RAG는 만들 수 있다. 그런데 현장 RAG에서는 대개 아래 같은 추가 요구가 금방 생긴다.

| 상황 | 체인으로 해결 가능? | 그래프가 풀어주는 방식 |
|---|---|---|
| 검색 결과가 질문과 무관할 때 다른 경로로 가고 싶다 | ❌ | 조건부 엣지 |
| 답변의 근거(관련성)를 검증하고 싶다 | △ | 독립 노드로 분리 |
| 재검색·재시도 같은 **루프**가 필요하다 | ❌ | 노드 간 역방향 엣지 |
| 여러 단계가 같은 데이터를 읽고 쓴다 | △ | 명시적 State |

03에서 익힌 **State · Node · Edge** 가 바로 이 지점에서 빛을 본다. 이 노트북에서는 두 단계로 쌓는다 — 먼저 단순한 **Naive RAG** 를 짠 다음, 거기에 **관련성 체크** 를 끼워 넣는다.

> 💡 **한 줄 요약**: 분기·반복·공유 상태 세 가지 중 하나라도 필요하면 LCEL 체인 대신 LangGraph로 옮겨간다.

## 2. Naive RAG 흐름 스케치

가장 단순한 형태는 단 두 단계다 — **문서를 찾고, 그 문서로 답변을 만든다**.

```
START ─▶ retrieve ─▶ llm_answer ─▶ END
```

두 노드가 함께 읽고 쓰는 값은 세 가지다.
- `question` — 사용자의 질문
- `context`   — 검색된 문서들을 이어 붙인 문자열
- `answer`    — LLM이 만든 최종 답변

이 세 값이 바로 **State** 다 — 두 노드가 공유하는 데이터 계약.

In [ ]:
# 코퍼스 로드 → 청크 분할 → FAISS 인덱스 → retriever 준비
# 03·05와 동일한 전처리 파라미터를 써서 결과 비교가 쉬워지도록 맞춘다.
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

docs = load_any('../data/corpus_ko.txt')
chunks = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40).split_documents(docs)
vectordb = FAISS.from_documents(chunks, get_embeddings())
retriever = vectordb.as_retriever(search_kwargs={'k': 3})

llm = get_chat_model(temperature=0.0)
print(f'코퍼스 청크 수: {len(chunks)}, retriever k=3 준비 완료')

## 3. State 정의 — 무엇을 노드 사이로 흘려야 할까?

State에 넣는 기준은 단순하다 — **여러 노드가 읽거나 써야 하는 값**이면 넣고, 한 노드 안에서만 쓰는 임시 변수는 넣지 않는다.

- `question` — retrieve 노드가 **읽고**, llm_answer 노드도 프롬프트에 넣으려고 **읽는다**.
- `context`  — retrieve가 **쓰고**, llm_answer가 **읽는다**.
- `answer`   — llm_answer가 **쓴다**.

세 필드 모두 "여러 노드가 쓰거나 읽는" 값이므로 State에 넣는 것이 맞다.

In [ ]:
from typing import TypedDict

class GraphState(TypedDict):
    question: str   # 사용자 질문
    context: str    # 검색된 문서를 이어 붙인 문자열
    answer: str     # LLM이 생성한 최종 답변

# TypedDict는 런타임에서 그냥 dict다 — 초기 값도 딕셔너리로 넣어주면 된다
sample: GraphState = {'question': '', 'context': '', 'answer': ''}
print(sample, type(sample))

## 4. 노드 정의 — 검색 노드와 답변 노드

노드 하나가 맡는 일을 한 줄로 정리하면:

- `retrieve_document(state)` — *"질문에 어울리는 문서를 찾아 `context`에 넣는다."*
- `llm_answer(state)`       — *"`context`만 보고 답변을 써서 `answer`에 넣는다."*

노드 하나가 "검색 + 답변"을 다 해버리면 나중에 *"검색만 다시 하고 싶다"* 나 *"답변을 만들기 전에 검증하고 싶다"* 같은 요구를 끼워 넣을 수 없다. 그래서 03에서 강조했던 **한 노드 한 역할** 원칙이 여기서 위력을 발휘한다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- 노드 1: 검색 ---
def retrieve_document(state: GraphState) -> dict:
    '''질문으로 문서를 찾아 context 문자열을 만들어 반환한다.'''
    q = state['question']
    hits = retriever.invoke(q)
    joined = '\n\n'.join(d.page_content for d in hits)
    return {'context': joined}

# --- 노드 2: 답변 생성 ---
answer_prompt = ChatPromptTemplate.from_template(
    '다음 컨텍스트만 사용해 한국어로 간결하게 답하라. '
    '컨텍스트에 근거가 없으면 "근거 없음"이라고 답하라.\n\n'
    '컨텍스트:\n{ctx}\n\n질문: {q}\n답변:'
)
answer_chain = answer_prompt | llm | StrOutputParser()

def llm_answer(state: GraphState) -> dict:
    '''context와 question을 프롬프트에 넣어 답변을 생성한다.'''
    reply = answer_chain.invoke({'ctx': state['context'], 'q': state['question']})
    return {'answer': reply}

print('노드 2개 정의 완료')

## 5. Naive RAG 그래프 조립 · 실행

노드를 모으고 순서를 지정한다.

```
StateGraph(GraphState)
  ├─ 노드: retrieve, llm_answer
  └─ 엣지: START → retrieve → llm_answer → END
```

In [ ]:
from langgraph.graph import StateGraph, START, END

g = StateGraph(GraphState)
g.add_node('retrieve', retrieve_document)
g.add_node('llm_answer', llm_answer)
g.add_edge(START, 'retrieve')
g.add_edge('retrieve', 'llm_answer')
g.add_edge('llm_answer', END)

naive_app = g.compile()

# --- 실행 — 코퍼스에 있는 주제로 질문 ---
init: GraphState = {
    'question': '전자금융거래에서 망분리는 어떤 원칙이고 왜 필요한가?',
    'context': '',
    'answer': '',
}
result = naive_app.invoke(init)

print('=== 검색된 context (앞 200자) ===')
print(result['context'][:200], '...')
print('\n=== 최종 답변 ===')
print(result['answer'])

## 6. 문제 인식 — Naive RAG가 흔들리는 순간

Naive RAG는 검색기를 무조건 믿는다. 그래서 검색기가 **질문과 거리가 먼 청크**를 상위로 넘겨주면, LLM은 그 청크 위에 그럴싸한 답변을 만들어낸다. 이게 바로 환각(hallucination)을 유발하는 전형적인 구조다.

> ⚠️ **주의**: "검색 성공 = 관련성 있음" 이 아니다. 검색은 *"벡터가 가까운 청크"* 를 뽑아줄 뿐, **질문에 실제로 답을 담은 청크**를 골라주지는 않는다.

해결책은 단순하다. **답변을 만들기 전에 검색 결과가 질문과 정말 관련 있는지 한 번 검사**하는 단계를 끼워 넣는다. 이걸 **Groundedness Check** 또는 **Relevance Check** 라고 부른다.

> 💡 **한 줄 요약**: 검색 결과를 살펴보는 가드 노드 하나만 추가하면 Naive RAG가 **자기교정** 의 첫 걸음을 내딛는다.

## 7. 관련성 체커 설계 — State에 `relevance`, 노드·라우터 추가

목표 흐름은 이렇게 바뀌어야 한다.

```
           ┌─ yes ─▶ llm_answer ─▶ END
retrieve ─▶ relevance_check
           └─ no  ─▶ retrieve (재검색)
```

필요한 변경은 세 가지다.
1. **State 확장**: 판정 결과를 어딘가에 적어둘 필드가 필요 → `relevance: str`.
2. **노드 추가**: `relevance_check` — LLM에게 "context가 question에 답할 근거를 갖추었는지" 물어 `yes`/`no` 한 단어만 받는다.
3. **조건부 엣지**: `relevance_check` 다음을 라우터 함수로 갈라놓는다.

> 📌 Groundedness Check는 "답변의 근거" 를 보는 체크다. 여기서는 가장 단순한 버전 — **question · context 쌍**을 보고 LLM이 직접 yes/no를 찍어주게 한다. 05에서는 문서 하나하나를 점수 매겨 필터링하는 조금 더 정교한 버전으로 확장된다.

In [ ]:
# State v2: relevance 필드 추가
class GraphState(TypedDict):   # 같은 이름으로 재정의
    question: str
    context: str
    answer: str
    relevance: str   # 'yes' / 'no'

print('GraphState v2 정의 완료 — relevance 필드 추가')

In [ ]:
# --- 노드 3: 관련성 체크 ---
relevance_prompt = ChatPromptTemplate.from_template(
    '아래 context가 주어진 question에 답할 근거를 담고 있는지 판단하라.\n'
    '관련 있으면 yes, 없으면 no 한 단어만 출력하라. 다른 설명은 붙이지 말 것.\n\n'
    'question: {q}\ncontext: {ctx}\n\nanswer(yes|no):'
)
relevance_chain = relevance_prompt | llm | StrOutputParser()

def relevance_check(state: GraphState) -> dict:
    '''context가 question에 답할 근거를 갖추었는지 yes/no로 판정한다.'''
    raw = relevance_chain.invoke({'q': state['question'], 'ctx': state['context']})
    verdict = 'yes' if raw.strip().lower().startswith('yes') else 'no'
    print(f'==== [RELEVANCE CHECK] {verdict} ====')
    return {'relevance': verdict}

# --- 라우터 함수 — State의 값을 그대로 라벨로 반환 ---
def is_relevant(state: GraphState) -> str:
    return state['relevance']   # 'yes' 또는 'no'

print('relevance_check 노드 + is_relevant 라우터 준비 완료')

## 8. 조건부 엣지가 포함된 v2 그래프 조립

핵심은 `add_conditional_edges` 한 줄이다. 라우터가 반환하는 라벨(`'yes'`/`'no'`)에 따라 그래프가 다음 노드를 고른다.

- `yes` → `llm_answer` (원래 답변 생성)
- `no`  → `retrieve`  (다시 검색 — 그런데 질문이 그대로니 결과가 바뀔까? 실험으로 확인할 것)

In [ ]:
g = StateGraph(GraphState)
g.add_node('retrieve', retrieve_document)
g.add_node('relevance_check', relevance_check)
g.add_node('llm_answer', llm_answer)

g.add_edge(START, 'retrieve')
g.add_edge('retrieve', 'relevance_check')
g.add_conditional_edges('relevance_check', is_relevant, {
    'yes': 'llm_answer',
    'no':  'retrieve',     # 관련성 없으면 다시 검색
})
g.add_edge('llm_answer', END)

app_v2 = g.compile()
print('v2 그래프 컴파일 완료 (조건부 엣지 포함)')

## 9. 성공 사례 — 코퍼스에 답이 있는 질문

`stream()` 으로 돌려 노드별 이벤트를 찍어본다. `retrieve → relevance_check → llm_answer` 순서로 각 단계에서 어떤 값이 State에 업데이트되는지 보일 것이다.

In [ ]:
init_ok: GraphState = {
    'question': '개인정보보호법에서 개인정보는 무엇으로 정의되는가?',
    'context': '', 'answer': '', 'relevance': '',
}

print('=== stream 이벤트 ===')
for event in app_v2.stream(init_ok):
    for node_name, update in event.items():
        preview = {k: (v[:60] + '...' if isinstance(v, str) and len(v) > 60 else v) for k, v in update.items()}
        print(f'[{node_name}] {preview}')

# 완성된 최종 state는 invoke로 한 번 더 받는다 (stream은 각 이벤트만 돌려주기 때문)
final_state = app_v2.invoke(init_ok)
print('\n=== 최종 답변 ===')
print(final_state['answer'])

## 10. 실패 사례 — 코퍼스에 없는 질문을 던지면?

코퍼스에 없는 주제 — 예를 들어 *"오늘 미국 다우 지수는 얼마였는가?"* 같은 질문을 넣으면, retriever는 그래도 *"벡터상 그나마 가까운"* 청크 몇 개를 가져온다. `relevance_check`는 당연히 `no` 로 판정하고, 그래프는 다시 `retrieve` 로 돌아간다.

그런데 주의해야 할 점 — **질문도 그대로, 인덱스도 그대로**다. 다시 검색해도 같은 청크가 나온다 → `relevance_check` 도 또 `no` → 또 재검색 → … 무한 루프.

LangGraph는 이런 상황에 대비해 **재귀 한계(`recursion_limit`)** 를 제공하며, 이 한도를 넘으면 `GraphRecursionError` 를 던진다. 아래에서 일부러 낮은 한계(6)를 주어 **에러를 재현** 해 본다.

In [ ]:
from langchain_core.runnables import RunnableConfig
from langgraph.errors import GraphRecursionError
import uuid

init_bad: GraphState = {
    'question': '오늘 미국 다우 지수와 애플 주가가 어떻게 움직였는지 알려줘.',
    'context': '', 'answer': '', 'relevance': '',
}
config = RunnableConfig(recursion_limit=6, configurable={'thread_id': str(uuid.uuid4())})

try:
    app_v2.invoke(init_bad, config=config)
except GraphRecursionError as e:
    print('\n>>> 예상대로 재귀 한계에서 막혔다.')
    print(f'GraphRecursionError: {e}')

## 11. 왜 무한 루프가 되는가 — 그리고 해결의 방향

이유는 허무할 정도로 단순하다.

> 다시 검색을 하러 돌아갈 때 **질문을 바꿔서 넘기지 않았다.** 같은 질문 · 같은 인덱스 → 같은 청크가 나올 수밖에 없다.

해결의 방향은 크게 두 갈래다.

| 전략 | 핵심 아이디어 | 다루는 노트북 |
|---|---|---|
| **쿼리 재작성 (Self-RAG)** | `no` → LLM이 질문을 다듬어 새 키워드로 바꾼 뒤 재검색 | 05 |
| **쿼리 복잡도 라우팅 (Adaptive RAG)** | 질문 유형을 먼저 분류해 아예 다른 경로로 | 06 |

> 📌 **여기서 멈춘다.** 이 노트북은 "관련성 체크를 조건부 엣지로 연결하기" 까지가 목표다. 무한 루프를 실제로 깨는 일은 05 Self-RAG에 맡긴다.

## 다음 단계

| 노트북 | 이 노트북과의 연결 |
|---|---|
| `05_self_rag_langgraph.ipynb` | `rewrite_query` 노드를 끼워 **no → 질문 갱신 → retrieve** 로 바꿔, 바로 위 무한 루프를 깬다 |
| `06_adaptive_rag.ipynb` | 질문 복잡도를 먼저 분류해 **아예 다른 경로** 로 보낸다 |

한 줄로 요약하면:

> **State에 `relevance` 하나 더 달고 조건부 엣지를 꽂은 순간, 그래프는 자기교정을 시작한다.** 지금 남은 질문은 단 하나 — *"그럼 질문은 어떻게 바꿔야 하는가?"* 그 답이 바로 05에 있다.